<!-- # SQLite Data Push and Quick View
Use this notebook to ingest files from `data/Val_data` into SQLite and inspect a few records without using terminal commands. -->

## Validation Dataset Prep (CSV -> JSON -> SQLite)
Step 1 converts `data/raw_data/test_datafile.csv` into a validation JSON inside `data/Val_data` with random 40 to 50 rows.
Sampling rule: keep more churned rows (`churned == 1`) with a target of 20 to 30 rows, and drop the `churned` key from JSON records.
Step 2 ingests the generated JSON into SQLite using the existing sync script (no `src` changes).

In [1]:
from pathlib import Path
import sqlite3
import subprocess
import sys
from zoneinfo import ZoneInfo

import pandas as pd

project_root = Path.cwd().resolve()

# Walk up from current directory until repository root is found.
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'app.py').exists() and (candidate / 'src').exists():
        project_root = candidate
        break
else:
    raise FileNotFoundError(f"Could not locate project root from {Path.cwd()}")

india_tz = ZoneInfo('Asia/Kolkata')
db_path = project_root / 'data' / 'SQLite_storage' / 'retention_sessions.db'
script_path = project_root / 'SQLlite_sync' / 'input_data_sqlite_sync.py'

print('Notebook CWD :', Path.cwd().resolve())
print('Project root :', project_root)
print('Raw data dir :', project_root / 'data' / 'raw_data')
print('DB path      :', db_path)
print('Sync script  :', script_path)

Notebook CWD : D:\Churn_Prediction\notebooks\SQLite
Project root : D:\Churn_Prediction
Raw data dir : D:\Churn_Prediction\data\raw_data
DB path      : D:\Churn_Prediction\data\SQLite_storage\retention_sessions.db
Sync script  : D:\Churn_Prediction\SQLlite_sync\input_data_sqlite_sync.py


In [2]:
import json
import random
from datetime import datetime

raw_csv_path = project_root / 'data' / 'raw_data' / 'test_datafile.csv'
val_dir = (project_root / 'data' / 'Val_data').resolve()
notebook_local_val_dir = (Path.cwd().resolve() / 'data' / 'Val_data').resolve()

# Safety guard: never use notebook-local data/Val_data.
if val_dir == notebook_local_val_dir and project_root.name.lower() != 'churn_prediction':
    raise RuntimeError('Invalid project_root resolved to notebook path. Re-run setup cell.')

val_dir.mkdir(parents=True, exist_ok=True)

# Randomize record counts within requested ranges.
total_target = random.randint(40, 50)
churned_target = random.randint(20, 30)
churned_target = min(churned_target, total_target)

df = pd.read_csv(raw_csv_path)
df.columns = [str(col).strip().lower().replace(' ', '_') for col in df.columns]

if 'customer_id' not in df.columns:
    raise ValueError("Required column 'customer_id' not found in input CSV.")
if 'churned' not in df.columns:
    raise ValueError("Required column 'churned' not found in input CSV.")

df['churned'] = pd.to_numeric(df['churned'], errors='coerce').fillna(0).astype(int)

churned_df = df[df['churned'] == 1]
non_churned_df = df[df['churned'] != 1]

if churned_df.empty:
    raise ValueError('No rows found where churned == 1. Cannot build requested validation sample.')

# Keep churned rows higher in sample as requested.
churned_n = min(len(churned_df), churned_target)
non_churned_n = total_target - churned_n
non_churned_n = min(len(non_churned_df), non_churned_n)

sample_parts = [churned_df.sample(n=churned_n, random_state=42)]
if non_churned_n > 0:
    sample_parts.append(non_churned_df.sample(n=non_churned_n, random_state=42))

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Remove churned from final validation JSON payload.
sample_df = sample_df.drop(columns=['churned'])
sample_df = sample_df.where(pd.notna(sample_df), None)

records = sample_df.to_dict(orient='records')
timestamp_tag = datetime.now(tz=india_tz).strftime('%Y%m%d_%H%M%S')
val_json_path = val_dir / f'churn_validation_sample_{timestamp_tag}.json'
val_json_path.write_text(json.dumps(records, ensure_ascii=True, indent=2, default=str), encoding='utf-8')

print('Raw CSV              :', raw_csv_path)
print('Validation dir       :', val_dir)
print('Validation JSON saved:', val_json_path)
print('Total sampled rows   :', len(records))
print('Sampled churned rows :', churned_n)
print('Sampled non-churned  :', len(records) - churned_n)
print('Contains churned key?:', 'churned' in (records[0] if records else {}))
# Ingest generated validation JSON from data/Val_data to SQLite via existing sync script.
if not script_path.exists():
    raise FileNotFoundError(f'Sync script not found: {script_path}')

cmd = [sys.executable, str(script_path), 'ingest', '--input-dir', str(project_root / 'data' / 'Val_data')]
ingest_result = subprocess.run(
    cmd,
    cwd=str(project_root),
    capture_output=True,
    text=True,
 )

print('Return code:', ingest_result.returncode)
if ingest_result.stdout.strip():
    print(ingest_result.stdout)
if ingest_result.stderr.strip():
    print('STDERR:\n' + ingest_result.stderr)

Raw CSV              : D:\Churn_Prediction\data\raw_data\test_datafile.csv
Validation dir       : D:\Churn_Prediction\data\Val_data
Validation JSON saved: D:\Churn_Prediction\data\Val_data\churn_validation_sample_20260726_020333.json
Total sampled rows   : 49
Sampled churned rows : 27
Sampled non-churned  : 22
Contains churned key?: False
Return code: 0
{
  "db_path": "D:\\Churn_Prediction\\data\\SQLite_storage\\retention_sessions.db",
  "input_dir": "D:\\Churn_Prediction\\data\\Val_data",
  "files_found": 1,
  "files_processed": 1,
  "files_skipped_unchanged": 0,
  "rows_total": 49,
  "inserted_total": 49,
  "updated_total": 0,
  "results": [
    {
      "file": "D:\\Churn_Prediction\\data\\Val_data\\churn_validation_sample_20260726_020333.json",
      "status": "ok",
      "rows": 49,
      "inserted": 49,
      "updated": 0
    }
  ]
}



In [3]:

# Quick check: latest registry entry after ingest.
with sqlite3.connect(db_path) as conn:
    registry_preview = pd.read_sql_query(
        """
        SELECT file_name, row_count, processed_at_time, status, error_message
        FROM input_file_registry
        ORDER BY processed_at_time DESC
        LIMIT 3
        """,
        conn,
    )

registry_preview

,file_name,row_count,processed_at_time,status,error_message
0,churn_validation_sample_20260726_020333.json,49,2026-07-26 02:03:36,ok,None


In [4]:
# current directory - find the repository root.
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'app.py').exists() and (candidate / 'src').exists():
        project_root = candidate
        break
else:
    raise FileNotFoundError(
        f"Could not locate the project root from {Path.cwd()}"
    )

india_tz = ZoneInfo('Asia/Kolkata')

def to_ist_text(value):
    if pd.isna(value):
        return value
    return pd.to_datetime(value).tz_localize(india_tz).strftime('%Y-%m-%d %H:%M:%S')

db_path = project_root / 'data' / 'SQLite_storage' / 'retention_sessions.db'
script_path = project_root / 'SQLlite_sync' / 'input_data_sqlite_sync.py'

print('Project root :', project_root)
print('DB path      :', db_path)
print('Sync script  :', script_path)
print('Script exists:', script_path.exists())

Project root : D:\Churn_Prediction
DB path      : D:\Churn_Prediction\data\SQLite_storage\retention_sessions.db
Sync script  : D:\Churn_Prediction\SQLlite_sync\input_data_sqlite_sync.py
Script exists: True


In [5]:
runtime_cwd = Path.cwd().resolve()

print('Notebook CWD :', runtime_cwd)
print('Project root :', project_root)
print('DB exists    :', db_path.exists())
print('Script exists:', script_path.exists())

is_ready = script_path.exists()
print('Ready to ingest:', is_ready)

Notebook CWD : D:\Churn_Prediction\notebooks\SQLite
Project root : D:\Churn_Prediction
DB exists    : True
Script exists: True
Ready to ingest: True


<!-- ## 1) Push (Ingest) Val_data into SQLite
This runs the same ingestion script used in the app and upserts by `customer_id`. -->

In [6]:
if not script_path.exists():
    raise FileNotFoundError(f'Sync script not found: {script_path}')

cmd = [sys.executable, str(script_path), 'ingest']

result = subprocess.run(
    cmd,
    cwd=str(project_root),
    capture_output=True,
    text=True
)

print('Return code:', result.returncode)

if result.stdout.strip():
    print(result.stdout)

if result.stderr.strip():
    print('STDERR:\n' + result.stderr)

Return code: 0
{
  "db_path": "D:\\Churn_Prediction\\data\\SQLite_storage\\retention_sessions.db",
  "input_dir": "D:\\Churn_Prediction\\data\\Val_data",
  "files_found": 1,
  "files_processed": 0,
  "files_skipped_unchanged": 1,
  "rows_total": 0,
  "inserted_total": 0,
  "updated_total": 0,
  "results": [
    {
      "file": "D:\\Churn_Prediction\\data\\Val_data\\churn_validation_sample_20260726_020333.json",
      "status": "skipped_unchanged",
      "rows": 0,
      "inserted": 0,
      "updated": 0
    }
  ]
}



<!-- ## 2) Check Available Tables and Row Counts -->

In [7]:
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """,
        conn
    )

counts = []
with sqlite3.connect(db_path) as conn:
    for table_name in tables['table_name'].tolist():
        row_count = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        counts.append({'table_name': table_name, 'row_count': row_count})

pd.DataFrame(counts)

,table_name,row_count
0,customer_validation_data,49
1,input_file_registry,1
2,output_records,0
3,session_events,1
4,sessions,1


In [8]:
# query = """
# DROP TABLE IF EXISTS input_file_registry;
# DROP TABLE IF EXISTS customer_validation_data;
# DROP TABLE IF EXISTS output_records;
# DROP TABLE IF EXISTS session_events;
# DROP TABLE IF EXISTS sessions;
# """

# with sqlite3.connect(db_path) as conn:
#     conn.executescript(query)

In [9]:
import sqlite3
import pandas as pd

with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type='table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name;
        """,
        conn,
    )

counts = []

with sqlite3.connect(db_path) as conn:
    for table_name in tables["table_name"]:
        # Get row count
        row_count = conn.execute(
            f"SELECT COUNT(*) FROM {table_name}"
        ).fetchone()[0]

        counts.append(
            {
                "table_name": table_name,
                "row_count": row_count,
            }
        )

        # Print table information
        print("\n" + "=" * 80)
        print(f"Table: {table_name}")
        print(f"Total Rows: {row_count}")
        print("=" * 80)

        # Print first 3 records
        sample_df = pd.read_sql_query(
            f"SELECT * FROM {table_name} LIMIT 3;",
            conn,
        )

        if sample_df.empty:
            print("No records found.")
        else:
            print(sample_df)

# Summary of all tables
counts_df = pd.DataFrame(counts)

print("\n" + "=" * 80)
print("Table Summary")
print("=" * 80)
print(counts_df)


Table: customer_validation_data
Total Rows: 49
  customer_id                                   source_file  \
0   TC-003303  churn_validation_sample_20260726_020333.json   
1   TC-004258  churn_validation_sample_20260726_020333.json   
2   TC-001153  churn_validation_sample_20260726_020333.json   

  source_modified_time     ingested_at_time  \
0  2026-07-26 02:03:33  2026-07-26 02:03:36   
1  2026-07-26 02:03:33  2026-07-26 02:03:36   
2  2026-07-26 02:03:33  2026-07-26 02:03:36   

                                         record_json  
0  {"customer_id": "TC-003303", "age": 45.2493746...  
1  {"customer_id": "TC-004258", "age": 63.3674373...  
2  {"customer_id": "TC-001153", "age": NaN, "gend...  

Table: input_file_registry
Total Rows: 1
                                           file_path  \
0  D:\Churn_Prediction\data\Val_data\churn_valida...   

                                      file_name  file_size  \
0  churn_validation_sample_20260726_020333.json      26376   

         m

<!-- ## 3) View a Few Customer Validation Records -->

In [10]:
query = """
SELECT *
FROM customer_validation_data
ORDER BY ingested_at_time DESC
LIMIT 5
"""
with sqlite3.connect(db_path) as conn:
    sample_df = pd.read_sql_query(query, conn)


sample_df

,customer_id,source_file,source_modified_time,ingested_at_time,record_json
0,TC-003303,churn_validation_sample_20260726_020333.json,2026-07-26 02:03:33,2026-07-26 02:03:36,"{""customer_id"": ""TC-003303"", ""age"": 45.2493746..."
1,TC-004258,churn_validation_sample_20260726_020333.json,2026-07-26 02:03:33,2026-07-26 02:03:36,"{""customer_id"": ""TC-004258"", ""age"": 63.3674373..."
2,TC-001153,churn_validation_sample_20260726_020333.json,2026-07-26 02:03:33,2026-07-26 02:03:36,"{""customer_id"": ""TC-001153"", ""age"": NaN, ""gend..."
3,TC-001684,churn_validation_sample_20260726_020333.json,2026-07-26 02:03:33,2026-07-26 02:03:36,"{""customer_id"": ""TC-001684"", ""age"": 46.8653892..."
4,TC-001366,churn_validation_sample_20260726_020333.json,2026-07-26 02:03:33,2026-07-26 02:03:36,"{""customer_id"": ""TC-001366"", ""age"": 39.6220564..."


<!-- ## 4) Expand JSON for One Customer
Enter a customer ID and inspect full validation JSON. -->

In [11]:
import json

customer_id = 'TC-000172'
query = """
SELECT customer_id, source_file, source_modified_time, ingested_at_time, record_json
FROM customer_validation_data
WHERE lower(customer_id) = lower(?)
LIMIT 1
"""

with sqlite3.connect(db_path) as conn:
    row = conn.execute(query, (customer_id,)).fetchone()

if row is None:
    print('Customer not found:', customer_id)
else:
    payload = {
        'customer_id': row[0],
        'source_file': row[1],
        'source_modified_time': row[2],
        'ingested_at_time': to_ist_text(row[3]),
        'record': json.loads(row[4])
    }
    print(json.dumps(payload, indent=2, ensure_ascii=True))

Customer not found: TC-000172


In [12]:
from collections import Counter, defaultdict
import json

# Show SQLite schema details for each table.
def inspect_sqlite_schema(connection):
    table_rows = connection.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """
    ).fetchall()

    schema_rows = []
    for table_row in table_rows:
        table_name = table_row[0]
        columns = connection.execute(f"PRAGMA table_info({table_name})").fetchall()
        for column in columns:
            schema_rows.append(
                {
                    "table": table_name,
                    "column": column[1],
                    "sqlite_type": column[2],
                    "not_null": bool(column[3]),
                    "primary_key": bool(column[5]),
                }
            )

    return pd.DataFrame(schema_rows)

# Show inferred JSON data types from the stored record_json column.
def inspect_json_value_types(connection, table_name="customer_validation_data", sample_limit=50):
    rows = connection.execute(
        f"""
        SELECT record_json
        FROM {table_name}
        ORDER BY ingested_at_time DESC
        LIMIT ?
        """,
        (sample_limit,),
    ).fetchall()

    field_types = defaultdict(Counter)
    field_examples = defaultdict(list)

    for row in rows:
        record = json.loads(row[0])
        for key, value in record.items():
            if value is None:
                value_type = "null"
            elif isinstance(value, bool):
                value_type = "bool"
            elif isinstance(value, int):
                value_type = "int"
            elif isinstance(value, float):
                value_type = "float"
            elif isinstance(value, list):
                value_type = "list"
            elif isinstance(value, dict):
                value_type = "dict"
            else:
                value_type = "str"

            field_types[key][value_type] += 1
            if len(field_examples[key]) < 3 and value not in field_examples[key]:
                field_examples[key].append(value)

    summary_rows = []
    for key in sorted(field_types):
        summary_rows.append(
            {
                "field": key,
                "observed_types": ", ".join(f"{name}:{count}" for name, count in field_types[key].most_common()),
                "sample_values": field_examples[key],
            }
        )

    return pd.DataFrame(summary_rows)

with sqlite3.connect(db_path) as conn:
    schema_df = inspect_sqlite_schema(conn)
    json_types_df = inspect_json_value_types(conn, sample_limit=50)

print("SQLite schema details (customer_id is the primary key in customer_validation_data; session columns are normalized):")
display(schema_df)

primary_key_df = schema_df[schema_df["primary_key"] == True]
print("\nPrimary key columns (for reference):")
display(primary_key_df)

print("\nInferred JSON data types from record_json:")
display(json_types_df)

SQLite schema details (customer_id is the primary key in customer_validation_data; session columns are normalized):


,table,column,sqlite_type,not_null,primary_key
0,customer_validation_data,customer_id,TEXT,False,True
1,customer_validation_data,source_file,TEXT,True,False
2,customer_validation_data,source_modified_time,TEXT,False,False
3,customer_validation_data,ingested_at_time,TEXT,True,False
4,customer_validation_data,record_json,TEXT,True,False
5,input_file_registry,file_path,TEXT,False,True
6,input_file_registry,file_name,TEXT,True,False
7,input_file_registry,file_size,INTEGER,True,False
8,input_file_registry,modified_time,TEXT,True,False
9,input_file_registry,sha256,TEXT,True,False



Primary key columns (for reference):


,table,column,sqlite_type,not_null,primary_key
0,customer_validation_data,customer_id,TEXT,False,True
5,input_file_registry,file_path,TEXT,False,True
14,output_records,id,INTEGER,False,True
21,session_events,id,INTEGER,False,True
29,sessions,session_id,TEXT,False,True



Inferred JSON data types from record_json:


,field,observed_types,sample_values
0,age,float:49,"[45.24937463, 63.36743732, nan]"
1,avg_monthly_gb_used,float:49,"[7.21, 6.68, 14.92]"
2,avg_monthly_minutes,float:49,"[273.9, 312.1, 336.0]"
3,contract_type,str:49,"[Month-to-month, One year, Two year]"
4,customer_id,str:49,"[TC-003303, TC-004258, TC-001153]"
5,gender,"str:48, null:1","[Female, Male, None]"
6,internet_service,"str:43, null:6","[Fiber optic, None, DSL]"
7,last_interaction_date,str:49,"[18-05-2024, 30-06-2024, 12-04-2024]"
8,monthly_charges,float:49,"[95.33, 97.07, 63.73]"
9,num_additional_services,int:49,"[5, 3, 2]"


In [13]:
query = """
SELECT file_name, file_path, modified_time, row_count, processed_at_time, status, error_message
FROM input_file_registry
ORDER BY processed_at_time DESC
LIMIT 5
"""
with sqlite3.connect(db_path) as conn:
    sample_df = pd.read_sql_query(query, conn)


sample_df[['file_name', 'file_path', 'modified_time', 'row_count', 'processed_at_time', 'status', 'error_message']]

,file_name,file_path,modified_time,row_count,processed_at_time,status,error_message
0,churn_validation_sample_20260726_020333.json,D:\Churn_Prediction\data\Val_data\churn_valida...,2026-07-26 02:03:33,49,2026-07-26 02:03:36,ok,None


In [14]:
with sqlite3.connect(db_path) as conn:
    session_columns = {row[1] for row in conn.execute("PRAGMA table_info(sessions)").fetchall()}

    started_at_column = (
        "started_at"
        if "started_at" in session_columns
        else "started_at_utc"
        if "started_at_utc" in session_columns
        else None
    )
    last_activity_column = (
        "last_activity_at"
        if "last_activity_at" in session_columns
        else "last_activity"
        if "last_activity" in session_columns
        else "last_activity_atc"
        if "last_activity_atc" in session_columns
        else None
    )

    select_parts = ["session_id", "status"]
    if started_at_column:
        select_parts.append(f"{started_at_column} AS started_at")
    if last_activity_column:
        select_parts.append(f"{last_activity_column} AS last_activity_at")
    select_parts.extend(["expires_at", "close_reason"])

    order_by_column = last_activity_column or started_at_column or "expires_at"
    query = f"""
SELECT
    {',\n    '.join(select_parts)}
FROM sessions
ORDER BY {order_by_column} DESC
"""
    sample_df = pd.read_sql_query(query, conn)

sample_df

,session_id,status,started_at,last_activity_at,expires_at,close_reason
0,RET-BFB8709598,active,2026-07-25T20:33:01+00:00,2026-07-25T20:33:01+00:00,2026-07-25T21:18:01+00:00,None
